## 은닉층 1개 = 직선 2개를 만든다는 의미 뉴런 2개
 - 입력 -> 은닉층(2개) -> 출력
 - z1 = w11\*x1 + w12\*x2 ... + b1
 - z2 = w21\*x1 + w22\*x2 ... + b2

In [ ]:
import numpy as np
np.random.seed(42)
# xor
X = np.array([
    (0,0),(1,1),(1,0),(0,1)
])
y = np.array([0,0,1,1])

# 활성화함수로 계단함수
def step(z):
   return np.where(z>=0, 1, 0) # 2차원이여서 이렇게 만들어줘야함

# 최초가중치
w1= np.random.randn(2)
w2= np.random.randn(2)
b1, b2 = np.random.randn(2)
print(f'최초 가중치 : {w1,b1}')
print(f'최초 가중치 : {w2,b2}')

# 은닉층 계산
z1 = np.dot(X,w1) + b1
z2 = np.dot(X,w2) + b2

print(f'은닉층 선형결합')
print(f'z1 : {z1}')
print(f'z2 : {z2}')

# 활성화 함수
h1 = step(z1)
h2 = step(z2)

print(f'은닉층 출력')
print(f'h1 : {h1}')
print(f'h2 : {h2}')

# 출력층
v = np.random.randn(2)
b_out = np.random.randn()
print(f'출력층 가중치 v : {v} b_out : {b_out}')

# 은닉층 묶기
H = np.column_stack((h1,h2))
# y_hat = v1*h1 + v2+h2 +b_out
z_out = np.dot(H,v) + b_out

print(f'출력층 선형결합')
print(f'z_out : {z_out}')

# 최종 출력
y_pred = step(z_out)

print(f'최종출력 : {y_pred}')

최초 가중치 : (array([ 0.49671415, -0.1382643 ]), np.float64(-0.23415337472333597))
최초 가중치 : (array([0.64768854, 1.52302986]), np.float64(-0.23413695694918055))
은닉층 선형결합
z1 : [-0.23415337  0.12429648  0.26256078 -0.37241768]
z2 : [-0.23413696  1.93658144  0.41355158  1.2888929 ]
은닉층 출력
h1 : [0 1 1 0]
h2 : [0 1 1 1]
출력층 가중치 v : [1.57921282 0.76743473] b_out : -0.4694743859349521
출력층 선형결합
z_out : [-0.46947439  1.87717316  1.87717316  0.29796034]
최종출력 : [0 1 1 1]


## 퍼셉트론의 업데이트 핵심
- error = y - y_hat
- w += lr\*error\*xi
- b += lr\*error

- 오차 -> 바로 가중치 수정 why 층이 하나고 이 층의 출력이 바로 결과

## 다중퍼셉트론(MLP)
 - 입력 -> 은닉층(h1, h2) -> 출력
 - 출력 오차 -> 출력층 수정 -> 은닉층도 수정

 - z1 = w1 * X + b1
 - h1 = step(z1)
 - z2 = w2 * X + b2
 - h2 = step(z2)
 - z = v1*h1 + v2*h2 + b


 ## backward(역전파)
 ### 출력층 업데이트
 - v += lr * error * h 

 ### 은닉층 업데이트
 - w += lr * error * X

In [ ]:
import numpy as np

np.random.seed(42)

# XOR 데이터
X = np.array([
    [0,0],
    [1,1],
    [0,1],
    [1,0]
])

y = np.array([[0],[0],[1],[1]])

lr = 0.1

# =========================
# sigmoid 함수
# =========================

def sigmoid(z): # 출력을 연속적인 데이터로 해서 미분이 가능하게 하기위해 
    return 1 / (1 + np.exp(-z))

# 합성함수 미분
def sigmoid_derivative(a): # sigmoid를 미분하기 위해 a를 대신 사용 , 출력이 여러 단계를 거쳐서 만들기 떄문에 사용
    return a * (1 - a)

# =========================
# 가중치 초기화 
# =========================

# 입력 → 은닉층
W1 = np.random.randn(2,2)
b1 = np.random.randn(1,2)

# 은닉층 → 출력층
W2 = np.random.randn(2,1)
b2 = np.random.randn(1,1)

# =========================
# 학습 루프
# =========================

for epoch in range(5000): 

    for xi, yi in zip(X,y):

        xi = xi.reshape(1,2)
        yi = yi.reshape(1,1)

        # =================
        # Forward
        # =================

        z1 = np.dot(xi, W1) + b1 
        h = sigmoid(z1)

        z2 = np.dot(h, W2) + b2
        y_hat = sigmoid(z2)

        # =================
        # Error
        # =================

        error = yi - y_hat

        # =================
        # Backprop  : 출력층 오차(error)를 은닉층까지 나눠서 전달 하는 과정
        # =================

        d_output = error * sigmoid_derivative(y_hat) # 출력층의 오차를 입력층으로 미분한다는 것 = 기울기

        d_hidden = (
            d_output.dot(W2.T) # 결과를 보고 오차가 왜 생겼는지 분석하는 것
            * sigmoid_derivative(h) # 은닉층 자체의 활성화 민감도
        )

        # =================
        # 가중치 업데이트
        # =================

        W2 += lr * h.T.dot(d_output) # 내적연산이 가능하게 .T를 해줌, 가중치와 같은 shape 만들려고 , 행렬 곱 계산
        b2 += lr * d_output

        W1 += lr * xi.T.dot(d_hidden)
        b1 += lr * d_hidden

# =========================
# 테스트
# =========================

print("\n학습 후 결과")

for xi, yi in zip(X,y):

    xi = xi.reshape(1,2)

    h = sigmoid(np.dot(xi,W1)+b1)
    y_hat = sigmoid(np.dot(h,W2)+b2)

    pred = 1 if y_hat >= 0.5 else 0

    print(
        xi.flatten(), # 다차원 데이터를 1차원으로 보이게 출력하는 함수
        "예측:", pred,
        "실제:", yi[0]
    )


학습 후 결과
[0 0] 예측: 0 실제: 0
[1 1] 예측: 0 실제: 0
[0 1] 예측: 1 실제: 1
[1 0] 예측: 1 실제: 1
